### 0. Import and Load

In [ ]:
import pandas as pd
import missingno as msno
from sklearn.impute import SimpleImputer
import seaborn as sns
import matplotlib.pyplot as plt
import miceforest as mf
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import math

In [ ]:
!curl -L -O https://github.com/mosesyhc/de300-2026sp/raw/refs/heads/main/homework/T_F41SCHEDULE_B43_with_missing.zip
inventory = pd.read_csv('T_F41SCHEDULE_B43_with_missing.zip')

### 1. Investigate Missing Data and Impute

In [ ]:
# Inspect overall missingness
msno.bar(inventory)
msno.matrix(inventory)

In [ ]:
def inspect_missing(df, col, other_cols):
    """Print the rows where the target column is missing, and looks at other columns of interest in those rows."""
    missing = df[df[col].isnull()]

    print(missing.count())
    for c in other_cols:
        print()
        print(missing[c].value_counts())
        print()

In [ ]:
# Inspect column missingness individually
inspect_missing(inventory, 'CARRIER', ['CARRIER_NAME'])
inspect_missing(inventory, 'CARRIER_NAME', ['CARRIER'])
inspect_missing(inventory, 'AIRLINE_ID', ['CARRIER'])
inspect_missing(inventory, 'MANUFACTURE_YEAR', ['YEAR', 'CARRIER'])
inspect_missing(inventory, 'NUMBER_OF_SEATS', ['YEAR', 'CARRIER'])
inspect_missing(inventory, 'CAPACITY_IN_POUNDS', ['YEAR'])

In [ ]:
# Show differences in correlation matrices before and after dropping rows with missing values in NUMBER_OF_SEATS
seats = inventory.dropna(subset='NUMBER_OF_SEATS', how='any')

corr = inventory.select_dtypes('number').corr()
corr_seats = seats.select_dtypes('number').corr()

corr - corr_seats

In [ ]:
def constant_impute(df, col, fill, condition=None) -> None:
    """Fill missing values in a column with constant imputation. Optionally takes a condition to only impute rows fulfilling that condition."""
    
    imputer = SimpleImputer(strategy='constant')
    imputer.fill_value = fill
    if condition:
        df.loc[df[condition[0]] == condition[1], col] = imputer.fit_transform(df.loc[df[condition[0]] == condition[1], [col]]).ravel()
    else:
        df[col] = imputer.fit_transform(df[[col]]).ravel()

In [ ]:
def predictive_mean_matching(df, num_datasets, num_candidates, iteration):
    """Predictive mean matching imputation on the given dataset. Returns a list of multiple imputed datasets."""
    
    kernel = mf.ImputationKernel(
        df.select_dtypes('number'),
        num_datasets=num_datasets,
        mean_match_candidates=num_candidates,
        random_state=1,
        save_all_iterations_data=False
    )
    
    kernel.mice(iteration)
    kernel.tune_parameters()
    inventory_pmm = kernel.complete_data
    return [inventory_pmm(dataset=i) for i in range(num_datasets)]

In [ ]:
def plot_impute_hist(df, num_datasets, pmm_datasets, col, binwidth):
    """Plot a histogram for a target column showing the imputed datasets in comparison with the original."""
    
    for k in range(num_datasets):
        sns.histplot(pmm_datasets[k][col], binwidth=binwidth, stat='probability', color='k', label='imputed', alpha=0.3)
    
    sns.histplot(df[col], binwidth=binwidth, stat='probability', color='purple', label='unimputed', alpha=0.2)
    
    plt.title(f'{num_datasets} Realizations of Imputed {col}')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
def pmm_impute(df, pmm_datasets, cols):
    """Modify the target dataframe to replace columns with the imputed pmm dataset mean."""
    
    pmm_mean_dataset = pd.concat(pmm_datasets).groupby(level=0).mean()
    for c in cols:
        df[c] = pmm_mean_dataset[c]

In [ ]:
def impute(df):
    """Impute selected target columns."""
    
    # Impute carrier, carrier_name, and airline_id columns with a constant imputer
    constant_impute(df, 'CARRIER', 'NA')
    constant_impute(df, 'CARRIER_NAME', 'Lynx Aviation d/b/a Frontier Airlines', ('CARRIER', 'L4'))
    constant_impute(df, 'AIRLINE_ID', 21217.0, ('CARRIER', 'L4'))
    
    # Create the pmm imputation with the following parameters
    num_datasets = 5
    num_candidates = 3
    iteration = 5
    pmm_datasets = predictive_mean_matching(df, num_datasets, num_candidates, iteration)
    
    # Finally modify the inventory dataset to replace number_of_seats and capacity_in_pounds with imputed values
    pmm_impute(df, pmm_datasets, ['NUMBER_OF_SEATS', 'CAPACITY_IN_POUNDS'])
    return num_datasets, pmm_datasets

In [ ]:
num_datasets, pmm_datasets = impute(inventory)

In [ ]:
# Visualize the imputations
plot_impute_hist(inventory, num_datasets, pmm_datasets, 'NUMBER_OF_SEATS', 10)
plot_impute_hist(inventory, num_datasets, pmm_datasets, 'CAPACITY_IN_POUNDS', 50000)

In [ ]:
# Confirm that missingness is removed in the target columns
msno.bar(inventory)

### 2. Transform and Standardize

In [ ]:
# Inspect the columns
for col in ['MANUFACTURER', 'MODEL', 'AIRCRAFT_STATUS', 'OPERATING_STATUS']:
    print(inventory[col].value_counts())
    print()

In [ ]:
def transform_uppercase(df, col):
    """Capitalize all letters in the column."""
    
    df[col] = df[col].str.upper()

In [ ]:
def transform_replace(df, col, option, remove_val, replace_val):
    """Replace target values in a column with given values. Can take 3 options: replace prefix, suffix, or any occurrances."""
    
    for i in range(len(remove_val)):
        if option == 'any':
            df[col] = df[col].str.replace(remove_val[i], replace_val[i], regex=True)
        
        elif option == 'prefix':
            mask = df[col].str.startswith(remove_val[i], na=False)
            df.loc[mask, col] = df.loc[mask, col].str.replace(f'^{remove_val[i]}({replace_val[i]})?', replace_val[i], regex=True)
            
        elif option == 'suffix':
            mask = df[col].str.endswith(remove_val[i], na=False)
            df.loc[mask, col] = df.loc[mask, col].str.replace(f'{remove_val[i]}$', replace_val[i], regex=True)

In [ ]:
def standardize_manufacturer(df):
    """Standardize the manufacturer column to have all uppercase entries, and remove common suffix and prefix."""
    
    # Change all entries to uppercase
    transform_uppercase(inventory, 'MANUFACTURER')
    
    # Remove common suffix and prefix
    transform_replace(inventory, 'MANUFACTURER', 'prefix', ["THE"], [""])
    transform_replace(inventory, 'MANUFACTURER', 'suffix', ["LTD", "INC", "CO", "COMPANY", "INDUSTRIES"], ["", "", "", "", ""])

In [ ]:
def standardize_model(df):
    """Standardize the model column to have all uppercase entries, and remove all hyphens."""
    
    # Change all entries to uppercase
    transform_uppercase(inventory, 'MODEL')
    
    # Remove hyphens
    transform_replace(inventory, 'MODEL', 'any', ["-"], [""])
    
    # Update dataset to use abbreviation for the most common manufacturers
    transform_replace(inventory, 'MODEL', 'prefix', ['BOEING', 'AIRBUS', 'EMBRAER'], ['B', 'A', 'EMB'])

In [ ]:
def standardize(df):
    """Standardize selected target columns."""
    
    standardize_manufacturer(df)
    standardize_model(df)
    transform_uppercase(df, 'AIRCRAFT_STATUS')
    transform_uppercase(df, 'OPERATING_STATUS')

In [ ]:
standardize(inventory)

In [ ]:
# Inspect the columns again
for col in ['MANUFACTURER', 'MODEL', 'AIRCRAFT_STATUS', 'OPERATING_STATUS']:
    print(inventory[col].value_counts())
    print()

### 3. Remove Remaining Missing Data

In [ ]:
def remove_remaining_missing(df):
    """Drop all remaining missing data and print the number of rows left."""
    
    df.dropna(how='any', inplace=True)
    print(df.shape)

In [ ]:
remove_remaining_missing(inventory)

In [ ]:
inventory.isnull().value_counts()

### 4. Transform and Derive Variables

In [ ]:
def plot_compare_hist(df, col):
    """Plot a histogram of the given variable."""
    
    sns.histplot(data=df, x=col, stat='probability')
    plt.title(f'Distribution of {col}')
    plt.show()

In [ ]:
# Histogram visualization before transformation
plot_compare_hist(inventory, 'NUMBER_OF_SEATS')
plot_compare_hist(inventory, 'CAPACITY_IN_POUNDS')

In [ ]:
def derive(df, cols):
    """Perform boxcox transformation on the two target columns."""
    
    for c in cols:
        df[f'{c}_BOXCOX'], _ = stats.boxcox(df[c].apply(lambda x: x + 1)) # need every entry to be positive

In [ ]:
derive(inventory, ['NUMBER_OF_SEATS', 'CAPACITY_IN_POUNDS'])

In [ ]:
# Histogram visualization after transformation
plot_compare_hist(inventory, 'NUMBER_OF_SEATS_BOXCOX')
plot_compare_hist(inventory, 'CAPACITY_IN_POUNDS_BOXCOX')

### 5. Feature Engineering

In [ ]:
def feature_engineering(df):
    """Create a new categorical column size based on the number of seats."""
    
    df['SIZE'] = pd.qcut(df['NUMBER_OF_SEATS'], q=4, labels=['SMALL', 'MEDIUM', 'LARGE', 'XLARGE'])

In [ ]:
def plot_bar(df, col):
    """Calculate and plot the proportion of target column by size."""
    
    # Calculate and display the proportions
    counts = df.groupby(['SIZE', col]).size().reset_index(name="count")
    counts['total'] = counts.groupby('SIZE')['count'].transform('sum')
    counts['proportion'] = counts['count'] / counts['total']
    print(counts)

    # Plot the proportions
    sns.barplot(data=counts, x='SIZE', y='proportion', hue=col)
    plt.title(f'Proportion by SIZE and {col}')
    plt.show()

In [ ]:
feature_engineering(inventory)

In [ ]:
# Display the proportions of the target two columns by size
plot_bar(inventory, 'OPERATING_STATUS')
plot_bar(inventory, 'AIRCRAFT_STATUS')

### 6. Modeling

In [ ]:
def preprocessing(df):
    """Preprocess the dataframe for modeling."""
    
    # Drop the non-numeric and boxcox columns
    df = df.select_dtypes('number').drop(columns=['NUMBER_OF_SEATS_BOXCOX', 'CAPACITY_IN_POUNDS_BOXCOX'])
    
    # Scale
    scaler = StandardScaler()
    df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
    
    # Train-test split and return the sub-datasets
    return train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
def build_model(train, test, option, col):
    """Train a model for a target column."""
    
    # Determine the type of model to use
    model = LinearRegression() if option == 'LINEAR' else RandomForestRegressor()
    
    # Identify the training and testing feature and label sets
    x_train, y_train = train.drop(columns=[col]), train[col]
    x_test, y_test = test.drop(columns=[col]), test[col]
    
    # Fit the model
    model.fit(x_train, y_train)
    
    # Training prediction and performance
    y_train_pred = model.predict(x_train)
    mse_train = mean_squared_error(y_train, y_train_pred)

    # Testing prediction and performance
    y_test_train = model.predict(x_test)
    mse_test = mean_squared_error(y_test, y_test_train)

    print("Training RMSE:", math.sqrt(mse_train))
    print("Testing RMSE:", math.sqrt(mse_test))


In [ ]:
def model(df):
    """Train and test models on target columns."""
    
    # Preprocess
    train, test = preprocessing(df)
    
    # Train models
    for t in ['LINEAR', 'RANDOM FOREST']:
        for c in ['NUMBER_OF_SEATS', 'CAPACITY_IN_POUNDS']:
            print(f'---RESULTS FOR {t} MODEL ON {c}---')
            build_model(train, test, t, c)
            print()

In [ ]:
model(inventory)